# Trichinella_Silver

In [1]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window
from pyspark.sql.types import IntegerType


# PARAMETERS

# Default. Quando correr pelo pipeline, este valor é substituído.
run_id = "manual"

print("Parâmetros recebidos pelo notebook:")
print("run_id:", run_id)

# 1. CARREGAR DADOS BRONZE
df_brz = spark.read.table("brz.trichinella")
df_corr = spark.read.table("brz.correspondencias_final")
df_siss = spark.read.table("brz.exploracoes_siss")

# 2. LIMPEZA BASE TRICHINELLA
# Deduplicação (Latest Snapshot)
win_tri = Window.partitionBy(
    "ncv",
    "datacolheita",
    "codigoexploracaoorigem",
    "especie_testada",
    "engorda_testados"
).orderBy(
    F.desc("meta_source_file_date"),
    F.desc("audit_brz_load_timestamp")
)

df_base = df_brz.withColumn("row_rank", F.row_number().over(win_tri)) \
    .filter(F.col("row_rank") == 1) \
    .drop("row_rank")

df_base = df_base.select(
    F.trim(F.col("ncv")).alias("ncv"),
    F.to_date(F.col("datacolheita")).alias("data_colheita"),
    F.trim(F.col("codigoexploracaoorigem")).alias("codigo_exploracao_origem"),
    F.col("engorda_testados").cast(IntegerType()).alias("engorda_testados"),
    F.col("reprodutores_testados").cast(IntegerType()).alias("reprodutores_testados"),
    F.col("tota_testados").cast(IntegerType()).alias("total_testados"),
    F.trim(F.col("especie_testada")).alias("especie_testada"),
    F.trim(F.col("regiao_do_matadouro")).alias("regiao_do_matadouro"),
    F.trim(F.col("matadouro")).alias("matadouro"),
    F.trim(F.col("pais_de_origem")).alias("pais_de_origem_original"),
    "meta_source_file",
    "meta_source_sheet",
    "meta_source_file_date"
).filter(
    F.upper(F.col("especie_testada")) == "SUÍNOS"
).filter(
    ~F.upper(F.col("codigo_exploracao_origem")).isin("NAN", "NULL", "")
)


# 3. LIMPEZA DE CÓDIGO (REGEX + CASE WHEN)
df_transform = df_base.withColumn(
    "marca_temp",
    F.trim(
        F.regexp_replace(
            F.col("codigo_exploracao_origem"),
            "(?i)(CANCELADO|ANULADA|INATIVA|INATIVO|SIPACE)",
            ""
        )
    )
)

df_transform = df_transform.withColumn(
    "codigo_reduzida_temp",
    F.when(
        F.col("marca_temp").startswith("PT"),
        F.substring(F.col("marca_temp"), 3, 100)
    ).otherwise(F.col("marca_temp"))
)

df_transform = df_transform.withColumn(
    "marca_temp",
    F.when(F.col("marca_temp") == "PTRB4C0PTRB66G", "PTRB4C0")
     .when(F.col("marca_temp") == "PTWF", "PTWF04B")
     .when(F.col("marca_temp") == "PTAV08", "PTAVA08")
     .when(F.col("marca_temp") == "RG5G8A", "RG5G8")
     .when(F.col("marca_temp") == "PTEAR49SS", "PTEAR49")
     .when(F.col("marca_temp") == "PTBX70", "PTBJX70")
     .when(F.col("marca_temp") == "PTA22T", "PTJA22T")
     .when(F.col("marca_temp") == "V50G", "PTVN50G")
     .when(F.col("marca_temp") == "PRRGV06", "PTRGV06")
     .when(F.col("marca_temp") == "9800953", "PT9U08V")
     .when(F.col("marca_temp") == "PYJAX74", "PTJAX74")
     .when(F.col("marca_temp") == "G24112011YD37C", "PTYD37C")
     .when(F.col("marca_temp") == "TF1Z4", "PTTF1Z4A")
     .when(F.col("marca_temp") == "KG1R6", "PTKG1R6A")
     .when(F.col("marca_temp") == "RS3F1", "PTRS3F1A")
     .when(F.col("marca_temp") == "RB04M", "PTRB04MA")
     .when(F.col("marca_temp").isin("PTRB8G5A"), "PTRB8G5")
     .when(F.col("marca_temp") == "PTRB5G2A", "PTRB5G2")
     .when(F.col("marca_temp") == "PTHE20TA", "PTHE20T")
     .when(F.col("marca_temp") == "VS67C", "PTVS67CA")
     .when(F.col("marca_temp") == "RWP48", "PTRWP48G")
     .when(F.col("marca_temp").isin("RY69T", "PTRY69TV"), "PTRY69T")
     .when(F.col("marca_temp") == "VS0AD", "PTVS0ADA")
     .when(F.col("marca_temp").isin("RY41H", "PTRY41HV"), "PTRY41H")
     .when(F.col("marca_temp") == "AAFG5", "PTAAFG58")
     .when(F.col("marca_temp").isin("RY45B", "PTRY45BV"), "PTRY45B")
     .when(F.col("marca_temp") == "R53F1", "PTR53F1A")
     .when(F.col("marca_temp") == "RG01S", "PTRG01SI")
     .when(F.col("marca_temp") == "RB542", "PTRB542A")
     .when(F.col("marca_temp") == "RG5G3", "PTRG5G3A")
     .when(F.col("marca_temp") == "HM02Z", "PTHM02ZA")
     .when(F.col("marca_temp") == "RB5G2", "PTRB5G2")
     .when(F.col("marca_temp") == "ES320820056601ES82OR402", "ES82OR402")
     .when(F.col("marca_temp").rlike(r"^\d{2}PT\d{5}$"), F.substring(F.col("marca_temp"), 3, 100))
     .when(
         F.col("marca_temp").startswith("PT") & (F.length(F.col("codigo_reduzida_temp")) == 6),
         F.substring(F.col("codigo_reduzida_temp"), 1, 5)
     )
     .otherwise(F.col("marca_temp"))
)

# Join com a Tabela de Correspondências
df_corr_clean = df_corr.select(
    F.trim(F.col("codigoexploracaoorigem")).alias("join_key"),
    F.trim(F.col("sugestao")).alias("sugestao_val")
).distinct()

df_corr_clean = df_corr_clean.withColumn(
    "sugestao_val",
    F.when(F.upper(F.col("sugestao_val")).isin("NAN", "NULL", ""), F.lit(None))
     .otherwise(F.col("sugestao_val"))
)

df_joined = df_transform.join(
    df_corr_clean,
    df_transform.marca_temp == df_corr_clean.join_key,
    "left"
)

df_pre_siss = df_joined.withColumn(
    "marca_base",
    F.coalesce(F.col("sugestao_val"), F.col("marca_temp"))
).withColumn(
    "marca_base",
    F.when(
        (F.upper(F.col("pais_de_origem_original")) == "PORTUGAL") & (F.length(F.col("marca_base")) == 5),
        F.concat(F.lit("PT"), F.col("marca_base"))
    ).otherwise(F.col("marca_base"))
).drop(
    "sugestao_val",
    "join_key",
    "marca_temp",
    "codigo_reduzida_temp"
)


# 4. CRIAR TABELA DE CORRESPONDÊNCIA SISS
df_keys = df_pre_siss.filter(F.col("marca_base").startswith("PT")) \
    .select("marca_base") \
    .distinct()

df_keys = df_keys.withColumn(
    "red_tri",
    F.substring(F.col("marca_base"), 3, 100)
)

df_keys = df_keys.withColumn(
    "par_tri",
    F.when(
        ~F.substring(F.col("red_tri"), 1, 5).rlike(r"^\d+$"),
        F.substring(F.col("red_tri"), 1, 5)
    ).otherwise(F.lit(None))
)

win_siss = Window.partitionBy("marca").orderBy(F.desc("meta_source_file_date"))

df_siss_latest = df_siss.withColumn("rn", F.row_number().over(win_siss)) \
    .filter(F.col("rn") == 1)

df_oficial = df_siss_latest.select(
    F.trim(F.col("marca")).alias("marca_oficial")
)

df_oficial = df_oficial.withColumn(
    "red_ofi",
    F.when(
        F.col("marca_oficial").startswith("PT"),
        F.substring(F.col("marca_oficial"), 3, 100)
    ).otherwise(F.col("marca_oficial"))
)

df_oficial = df_oficial.withColumn(
    "par_ofi",
    F.when(
        ~F.substring(F.col("red_ofi"), 1, 5).rlike(r"^\d+$"),
        F.substring(F.col("red_ofi"), 1, 5)
    ).otherwise(F.lit(None))
)

siss_exata = df_oficial.select("marca_oficial") \
    .dropDuplicates(["marca_oficial"])

siss_reduz = df_oficial.select("marca_oficial", "red_ofi") \
    .dropDuplicates(["red_ofi"])

siss_parci = df_oficial.filter(F.col("par_ofi").isNotNull()) \
    .select("marca_oficial", "par_ofi") \
    .dropDuplicates(["par_ofi"])

matched = df_keys.join(
    siss_exata,
    df_keys.marca_base == siss_exata.marca_oficial,
    "left"
).withColumnRenamed("marca_oficial", "m1")

matched = matched.join(
    siss_reduz,
    matched.red_tri == siss_reduz.red_ofi,
    "left"
).withColumnRenamed("marca_oficial", "m2").drop("red_ofi")

matched = matched.join(
    siss_parci,
    matched.par_tri == siss_parci.par_ofi,
    "left"
).withColumnRenamed("marca_oficial", "m3").drop("par_ofi")

matched = matched.withColumn(
    "arr",
    F.array_distinct(F.expr("filter(array(m1, m2, m3), x -> x is not null)"))
)

df_bridge = matched.withColumn(
    "marca_siss",
    F.array_join(F.col("arr"), "; ")
).withColumn(
    "flg_siss_multi",
    F.when(F.size(F.col("arr")) > 1, "Sim").otherwise("Não")
).select(
    "marca_base",
    "marca_siss",
    "flg_siss_multi"
)


# 5. SUBSTITUIÇÃO NA BASE PRINCIPAL
df_final = df_pre_siss.join(df_bridge, "marca_base", "left")

df_final = df_final.withColumn(
    "marca",
    F.when(
        F.col("marca_siss").isNotNull() & (F.col("marca_siss") != ""),
        F.col("marca_siss")
    ).otherwise(F.col("marca_base"))
)

df_final = df_final.withColumn(
    "codigo_reduzida",
    F.when(
        F.substring(F.col("marca"), 1, 2) == "PT",
        F.substring(F.col("marca"), 3, 100)
    ).otherwise(F.col("marca"))
).withColumn(
    "pais_de_origem",
    F.when(F.col("marca").startswith("PT"), "Portugal")
     .when(F.col("marca").rlike("^(ES|Es|es)"), "Espanha")
     .otherwise(F.col("pais_de_origem_original"))
).drop(
    "pais_de_origem_original",
    "marca_base",
    "marca_siss"
).filter(F.upper(F.col("pais_de_origem")) == "PORTUGAL")


# 6. AGREGAÇÃO FINAL E SAVE NA SILVER
df_aggregated = df_final.groupBy(
    "ncv",
    "data_colheita",
    "codigo_exploracao_origem",
    "marca",
    "codigo_reduzida",
    "regiao_do_matadouro",
    "matadouro",
    "pais_de_origem",
    "flg_siss_multi"
).agg(
    F.sum("engorda_testados").alias("total_engorda"),
    F.sum("reprodutores_testados").alias("total_reprodutores"),
    F.sum("total_testados").alias("total_testados"),
    F.concat_ws(" | ", F.collect_set("meta_source_file")).alias("meta_source_file"),
    # F.collect_set("meta_source_file").alias("meta_source_file"),
    F.max("meta_source_file_date").alias("meta_source_file_date")
)


df_silver = (
    df_aggregated
    .withColumn("audit_source_bronze", F.lit("brz.trichinella"))
    .withColumn("audit_silver_refresh_timestamp", F.current_timestamp())
    .withColumn("audit_run_id", F.lit(run_id))
)

spark.sql("CREATE SCHEMA IF NOT EXISTS slv")

df_silver.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("slv.trichinella")

print("Sucesso! Tabela Silver gerada já com as Marcas Oficiais do SISS substituídas e agregadas.")


StatementMeta(, 8b2fe449-9737-400a-a19d-72523bbdbff2, 3, Finished, Available, Finished, False)

Sucesso! Tabela Silver gerada já com as Marcas Oficiais do SISS substituídas e agregadas.


## Validação

In [11]:
# from pyspark.sql import functions as F

# # 1. Carregar a Tabela Silver
# table_name = "slv.trichinella"
# df_slv = spark.read.table(table_name)

# print(f"{'='*70}")
# print(f"EXPLORAÇÃO E VALIDAÇÃO DA CAMADA SILVER: {table_name}")
# print(f"{'='*70}\n")

# # --- Passo 1: Volume de Dados ---
# print("1. VOLUME TOTAL")
# total_rows = df_slv.count()
# print(f"Total de registos agregados: {total_rows}\n")

# # --- Passo 2: Qualidade de Dados (Valores Nulos) ---
# print("2. SAÚDE DAS CHAVES PRINCIPAIS (Contagem de Nulos)")
# df_slv.select([
#     F.count(F.when(F.col(c).isNull() | (F.col(c) == ""), c)).alias(c) 
#     for c in["ncv", "data_colheita", "codigo_exploracao_origem", "marca", "pais_de_origem"]
# ]).show()

# # --- Passo 3: Distribuição de País de Origem ---
# print("3. DISTRIBUIÇÃO DO PAÍS DE ORIGEM")
# # Valida se "Portugal", "Espanha" estão corretos, e mostra os valores originais que caíram no "Outro"
# df_slv.groupBy("pais_de_origem").count().orderBy(F.desc("count")).limit(10).show(truncate=False)

# # --- Passo 4: Validação do Motor SISS (Múltiplos Matches) ---
# print("4. VALIDAÇÃO DO MOTOR SISS: MARCAS MÚLTIPLAS")
# # Mostra as marcas que bateram certo com mais do que uma exploração oficial no SISS
# df_multi = df_slv.filter(F.col("flg_siss_multi") == "Sim")
# print(f"Registos com múltiplas correspondências no SISS: {df_multi.count()}")
# if df_multi.count() > 0:
#     df_multi.select("codigo_exploracao_origem", "marca", "total_testados_sum") \
#             .distinct().orderBy(F.desc("total_testados_sum")).limit(5).show(truncate=False)

# # --- Passo 5: Validação da Regra dos 5 Dígitos (Portugal) ---
# print("5. VALIDAÇÃO: REGRA DOS 5 DÍGITOS PARA PORTUGAL")
# # Verifica se as marcas originais com 5 dígitos levaram com o prefixo 'PT' na coluna final
# df_slv.filter(
#     (F.col("pais_de_origem") == "PORTUGAL") & 
#     (F.length(F.col("codigo_exploracao_origem")) == 5)
# ).select("codigo_exploracao_origem", "marca", "codigo_reduzida").distinct().limit(5).show()

# # --- Passo 6: Top 10 Explorações (Teste à Agregação) ---
# print("6. TOP 10 EXPLORAÇÕES POR VOLUME DE TESTES")
# df_slv.groupBy("marca", "pais_de_origem") \
#     .agg(F.sum("total_testados_sum").alias("Total_Animais_Testados")) \
#     .orderBy(F.desc("Total_Animais_Testados")) \
#     .limit(10) \
#     .show(truncate=False)

# # --- Passo 7: Preview Visual ---
# print("7. PREVIEW DOS DADOS FINAIS (Amostra)")
# display(df_slv.limit(10))

# print(f"\n{'='*70}")
# print("VALIDAÇÃO CONCLUÍDA")
# print(f"{'='*70}")


# from pyspark.sql import functions as F

# table_name = "slv.trichinella"
# df_slv = spark.read.table(table_name)

# print(f"{'='*70}")
# print(f"EXPLORAÇÃO E VALIDAÇÃO DA CAMADA SILVER: {table_name}")
# print(f"{'='*70}\n")

# print("1. VOLUME TOTAL")
# total_rows = df_slv.count()
# print(f"Total de registos agregados: {total_rows}\n")

# print("2. SAÚDE DAS CHAVES PRINCIPAIS")
# df_slv.select([
#     F.count(
#         F.when(
#             F.col(c).isNull() | (F.trim(F.col(c)) == ""),
#             c
#         )
#     ).alias(c)
#     for c in ["ncv", "data_colheita", "codigo_exploracao_origem", "marca", "pais_de_origem"]
# ]).show()

# print("3. DISTRIBUIÇÃO DO PAÍS DE ORIGEM")
# df_slv.groupBy("pais_de_origem") \
#     .count() \
#     .orderBy(F.desc("count")) \
#     .limit(10) \
#     .show(truncate=False)

# print("4. VALIDAÇÃO DO MOTOR SISS: MARCAS MÚLTIPLAS")
# df_multi = df_slv.filter(F.col("flg_siss_multi") == "Sim")
# multi_count = df_multi.count()

# print(f"Registos com múltiplas correspondências no SISS: {multi_count}")

# if multi_count > 0:
#     df_multi.select(
#         "codigo_exploracao_origem",
# #         "marca",
#         "total_testados_sum"
#     ).distinct() \
#      .orderBy(F.desc("total_testados_sum")) \
#      .limit(5) \
#      .show(truncate=False)

# print("5. VALIDAÇÃO: REGRA DOS 5 DÍGITOS PARA PORTUGAL")
# df_slv.filter(
#     (F.upper(F.col("pais_de_origem")) == "PORTUGAL") &
#     (F.length(F.col("codigo_exploracao_origem")) == 5)
# ).select(
#     "codigo_exploracao_origem",
#     "marca",
#     "codigo_reduzida"
# ).distinct().limit(5).show(truncate=False)

# print("6. TOP 10 EXPLORAÇÕES POR VOLUME DE TESTES")
# df_slv.groupBy("marca", "pais_de_origem") \
#     .agg(F.sum("total_testados_sum").alias("Total_Animais_Testados")) \
#     .orderBy(F.desc("Total_Animais_Testados")) \
#     .limit(10) \
#     .show(truncate=False)

# print("7. PREVIEW DOS DADOS FINAIS")
# display(df_slv.limit(10))

# print(f"\n{'='*70}")
# print("VALIDAÇÃO CONCLUÍDA")
# print(f"{'='*70}")

StatementMeta(, e3aa027f-b656-4c6a-a50e-e62aed6832ee, 4, Finished, Available, Finished, False)

EXPLORAÇÃO E VALIDAÇÃO DA CAMADA SILVER: slv.trichinella

1. VOLUME TOTAL


Total de registos agregados: 589901

2. SAÚDE DAS CHAVES PRINCIPAIS


+---+-------------+------------------------+-----+--------------+
|ncv|data_colheita|codigo_exploracao_origem|marca|pais_de_origem|
+---+-------------+------------------------+-----+--------------+
|  0|            0|                       0|    0|             0|
+---+-------------+------------------------+-----+--------------+

3. DISTRIBUIÇÃO DO PAÍS DE ORIGEM
+--------------+------+
|pais_de_origem|count |
+--------------+------+
|Portugal      |507419|
|Espanha       |81936 |
|Bélgica       |466   |
|Holanda       |38    |
|França        |33    |
|Alemanha      |8     |
|Dinamarca     |1     |
+--------------+------+

4. VALIDAÇÃO DO MOTOR SISS: MARCAS MÚLTIPLAS


Registos com múltiplas correspondências no SISS: 0
5. VALIDAÇÃO: REGRA DOS 5 DÍGITOS PARA PORTUGAL
+------------------------+-------+---------------+
|codigo_exploracao_origem|marca  |codigo_reduzida|
+------------------------+-------+---------------+
|RG93B                   |PTRG93B|RG93B          |
|JT4AX                   |PTJT4AX|JT4AX          |
|BNR36                   |PTBNR36|BNR36          |
|EA61T                   |PTEA61T|EA61T          |
|VZ09C                   |PTVZ09C|VZ09C          |
+------------------------+-------+---------------+

6. TOP 10 EXPLORAÇÕES POR VOLUME DE TESTES


+--------+--------------+----------------------+
|marca   |pais_de_origem|Total_Animais_Testados|
+--------+--------------+----------------------+
|PTTF1Z4A|Portugal      |1945341               |
|PTRB5G2 |Portugal      |937044                |
|PTVW73P |Portugal      |740876                |
|ptrb5g2a|Portugal      |338383                |
|PTRY04A |Portugal      |322030                |
|PTKG1R6A|Portugal      |297440                |
|PTRG99J |Portugal      |288322                |
|PTVR90G |Portugal      |279826                |
|PTVW69C |Portugal      |272066                |
|PTRB4G2 |Portugal      |256104                |
+--------+--------------+----------------------+

7. PREVIEW DOS DADOS FINAIS


SynapseWidget(Synapse.DataFrame, acd92d8b-461f-44ad-956d-9733e613afa0)


VALIDAÇÃO CONCLUÍDA


### gold test

In [11]:
# from pyspark.sql import functions as F

# print(f"{'='*80}")
# print("CONSTRUÇÃO DA TABELA DE FACTOS: gld.fact_controlo_sanitario")
# print(f"{'='*80}\n")


# # 1. CARREGAR TABELAS (SILVER E DIMENSÕES GOLD)
# df_slv_tri = spark.read.table("slv.trichinella")
# df_slv_rep = spark.read.table("slv.motivosreprovacao")  # <-- Nova tabela
# df_dim_exp = spark.read.table("gld.dim_exploracao_scd") 
# df_dim_mat = spark.read.table("gld.dim_matadouro")


# # 2. PREPARAÇÃO E AGREGAÇÃO: ABATES (TRICHINELLA)
# df_fact = df_slv_tri.withColumnRenamed("data_colheita", "data") \
#                     .withColumn("ncv", F.trim(F.regexp_replace(F.col("ncv"), r"(?i)\s*\(CANCELADO\)", "")))

# df_agg_tri = df_fact.groupBy("data", "marca", "ncv").agg(
#     F.sum("total_testados").alias("abatidos"),
#     F.first("meta_source_file").alias("meta_source_file"),
#     F.max("meta_source_file_date").alias("meta_source_file_date")
# )


# 3. PREPARAÇÃO E AGREGAÇÃO: REPROVAÇÕES
# # Renomear data e limpar NCV com a mesma regra para garantir o Join perfeito
# df_rep_prep = df_slv_rep.withColumnRenamed("data_controlo", "data") \
#                         .withColumn("ncv", F.trim(F.regexp_replace(F.col("ncv"), r"(?i)\s*\(CANCELADO\)", "")))

# df_agg_rep = df_rep_prep.groupBy("data", "marca", "ncv").agg(
#     F.sum("qtd_animais_reprovados").alias("reprovados")
# )


# # 4. CONSOLIDAÇÃO DA FACT TABLE & REGRAS DE NEGÓCIO
# # Left join com a base de abates
# df_base_fact = df_agg_tri.join(df_agg_rep, ["data", "marca", "ncv"], "left")

# # Regra: Nulos em reprovados passam a 0
# df_base_fact = df_base_fact.withColumn("reprovados", F.coalesce(F.col("reprovados"), F.lit(0)))

# # --- AVALIAÇÃO DE REGRA DE NEGÓCIO (Para o bloco de validação depois) ---
# # Em vez de apenas apagar, vamos contar primeiro as inconsistências
# linhas_inconsistentes = df_base_fact.filter(F.col("reprovados") > F.col("abatidos")).count()

# # Aplicação do Filtro: Manter apenas onde reprovados <= abatidos
# df_base_fact = df_base_fact.filter(F.col("reprovados") <= F.col("abatidos"))


# # 5. JOIN 1: DIMENSÃO EXPLORAÇÃO (SCD TIPO 2)
# condicao_exp = [
#     df_base_fact.marca == df_dim_exp.marca,
#     df_base_fact.data >= df_dim_exp.data_inicio,
#     df_base_fact.data <= df_dim_exp.data_fim
# ]

# df_join1 = df_base_fact.join(df_dim_exp, condicao_exp, "left") \
#     .select(
#         df_base_fact["*"],
#         df_dim_exp["SK_Historico"].alias("SK_Marca_Historico"),
#         df_dim_exp["SK_Geo_Exp"]
#     )


# # 6. JOIN 2: DIMENSÃO MATADOURO
# condicao_mat = [df_join1.ncv == df_dim_mat.ncv]

# df_join2 = df_join1.join(df_dim_mat, condicao_mat, "left") \
#     .select(
#         df_join1["*"],
#         df_dim_mat["SK_Matadouro"],
#         df_dim_mat["SK_Geo_Mat"]
#     )


# # 7. SELEÇÃO FINAL E AUDITORIA
# df_final = df_join2.select(
#     # --- Chaves Estrangeiras (Foreign Keys) ---
#     "SK_Marca_Historico",
#     "SK_Geo_Exp",
#     "SK_Matadouro",
#     "SK_Geo_Mat",
    
#     # --- Colunas de Negócio (Factos) ---
#     "data",
#     "abatidos",
#     "reprovados", # <-- Nova métrica adicionada
    
#     # --- Colunas Temporárias (Para validação) ---
#     F.col("marca").alias("marca_TO_DROP"),
#     F.col("ncv").alias("ncv_TO_DROP"),
    
#     # --- Auditoria e Metadados ---
#     "meta_source_file",
#     "meta_source_file_date",
#     F.lit("slv.trichinella | slv.motivosreprovacao").alias("audit_source_silver"),
#     F.current_timestamp().alias("audit_gold_refresh_timestamp")
# )


# # 8. GUARDAR NA GOLD E REPORT DE REGRAS
# spark.sql("CREATE SCHEMA IF NOT EXISTS gld")

# df_final.write \
#     .format("delta") \
#     .mode("overwrite") \
#     .option("overwriteSchema", "true") \
#     .saveAsTable("gld.fact_controlo_sanitario")

# print(f"Sucesso! Tabela de factos gerada com {df_final.count()} registos.")
# print(f"ATENÇÃO: {linhas_inconsistentes} linhas foram removidas porque 'reprovados > abatidos'.")

StatementMeta(, 628180d6-037f-41ee-aa61-49d4ba4fa427, 13, Finished, Available, Finished, False)

CONSTRUÇÃO DA TABELA DE FACTOS: gld.fact_controlo_sanitario

Sucesso! Tabela de factos gerada com 589791 registos.
ATENÇÃO: 9 linhas foram removidas porque 'reprovados > abatidos'.


In [12]:
# from pyspark.sql import functions as F

# print(f"{'='*80}")
# print("AUDITORIA DE QUALIDADE DA TABELA DE FACTOS E REGRAS DE NEGÓCIO")
# print(f"{'='*80}\n")

# # 1. Carregar as bases necessárias
# df_fact = spark.read.table("gld.fact_controlo_sanitario")
# total_linhas = df_fact.count()

# print(f"-> TOTAL DE LINHAS NA FACT TABLE FINAL: {total_linhas}\n")


# # PARTE A: INVESTIGAÇÃO DA REGRA DE REPROVADOS > ABATIDOS
# print("--- 1. INVESTIGAÇÃO DE INCONSISTÊNCIAS NOS LOTES (Reprovados > Abatidos) ---")
# # Para mostrar quais foram, recriamos rapidamente o join antes do filtro:
# df_tri = spark.read.table("slv.trichinella").withColumnRenamed("data_colheita", "data").withColumn("ncv", F.trim(F.regexp_replace(F.col("ncv"), r"(?i)\s*\(CANCELADO\)", "")))
# df_rep = spark.read.table("slv.motivosreprovacao").withColumnRenamed("data_controlo", "data").withColumn("ncv", F.trim(F.regexp_replace(F.col("ncv"), r"(?i)\s*\(CANCELADO\)", "")))

# # agg_tri = df_tri.groupBy("data", "marca", "ncv").agg(F.sum("total_testados").alias("abatidos"))
# agg_rep = df_rep.groupBy("data", "marca", "ncv").agg(F.sum("qtd_animais_reprovados").alias("reprovados"))

# inconsistentes = agg_tri.join(agg_rep, ["data", "marca", "ncv"], "left") \
#                         .withColumn("reprovados", F.coalesce(F.col("reprovados"), F.lit(0))) \
#                         .filter(F.col("reprovados") > F.col("abatidos"))

# qtd_inconsistentes = inconsistentes.count()
# print(f"Total de lotes com erros lógicos removidos da Fact Table: {qtd_inconsistentes}")

# # if qtd_inconsistentes > 0:
# #     print("\nExemplo dos piores casos (onde os reprovados superaram largamente os abatidos):")
#     # inconsistentes.withColumn("diferenca_impossivel", F.col("reprovados") - F.col("abatidos")) \
#                   .orderBy(F.desc("diferenca_impossivel")) \
#                   .show(10)



# PARTE B: VALIDAÇÃO DE MATCHING DE DIMENSÕES (CHAVES ESTRANGEIRAS)
# print("\n--- 2. AUDITORIA DE MATCH RATE (Chaves Estrangeiras) ---")

# # -- Explorações --
# com_id_exp = df_fact.filter(F.col("SK_Marca_Historico").isNotNull()).count()
# sem_id_exp = total_linhas - com_id_exp
# perc_exp = round((com_id_exp / total_linhas) * 100, 2) if total_linhas > 0 else 0

# print("\nA. LIGAÇÃO À DIMENSÃO EXPLORAÇÃO (SCD2):")
# print(f"   - Match Sucesso: {com_id_exp} ({perc_exp}%)")
# print(f"   - Match Falhado: {sem_id_exp}")

# if sem_id_exp > 0:
#     print("   -> Marcas que falharam o match ou estão fora das datas (Top 5):")
# #     df_fact.filter(F.col("SK_Marca_Historico").isNull()) \
# #            .groupBy("marca_TO_DROP").count().orderBy(F.desc("count")).limit(5).show()

# # -- Matadouros --
# com_id_mat = df_fact.filter(F.col("SK_Matadouro").isNotNull()).count()
# sem_id_mat = total_linhas - com_id_mat
# perc_mat = round((com_id_mat / total_linhas) * 100, 2) if total_linhas > 0 else 0

# print("\nB. LIGAÇÃO À DIMENSÃO MATADOURO:")
# print(f"   - Match Sucesso: {com_id_mat} ({perc_mat}%)")
# print(f"   - Match Falhado: {sem_id_mat}")

# if sem_id_mat > 0:
#     print("   -> Matadouros (NCVs) que não existem na dimensão (Top 5):")
#     df_fact.filter(F.col("SK_Matadouro").isNull()) \
#            .groupBy("ncv_TO_DROP").count().orderBy(F.desc("count")).limit(5).show()

# print(f"\n{'='*80}")

StatementMeta(, 628180d6-037f-41ee-aa61-49d4ba4fa427, 14, Finished, Available, Finished, False)

AUDITORIA DE QUALIDADE DA TABELA DE FACTOS E REGRAS DE NEGÓCIO

-> TOTAL DE LINHAS NA FACT TABLE FINAL: 589791

--- 1. INVESTIGAÇÃO DE INCONSISTÊNCIAS NOS LOTES (Reprovados > Abatidos) ---
Total de lotes com erros lógicos removidos da Fact Table: 9

Exemplo dos piores casos (onde os reprovados superaram largamente os abatidos):
+----------+--------------+----+--------+----------+--------------------+
|      data|         marca| ncv|abatidos|reprovados|diferenca_impossivel|
+----------+--------------+----+--------+----------+--------------------+
|2011-11-17|       PTRH53P|R 19|      15|       337|                 322|
|2012-02-22|       PTRG86B|R 29|     121|       315|                 194|
|2020-10-20|       PTRB8G3|R 93|      63|       169|                 106|
|2023-08-14|es360520229201|D 36|      17|       119|                 102|
|2023-03-31|ES320780061601|D 61|     110|       210|                 100|
|2020-04-17|ES360240256401|D 42|     220|       278|                  58|
|201